# Simple LLM Workflow 

Sequential Workflow :

In [9]:
from langgraph.graph import StateGraph , START , END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [10]:
load_dotenv()

True

In [11]:
import os

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [12]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    temperature=0
)

In [13]:
# Create a State : 

class LLMState(TypedDict):
    question : str
    answer : str


In [14]:
# Creating functions/task :

# Task1 :

def llm_qa(state : LLMState) -> LLMState:
    # extract the question from state :
    question = state['question']

    # form a prompt : 
    prompt = f'Answer the follwing quesetion {question}'

    # ask that question to the llm : 
    answer = model.invoke(prompt).content

    # update the answer in the state :
    state['answer'] = answer

    return state


In [15]:
# Create Graph : 
graph = StateGraph(LLMState)

# add nodes : 

graph.add_node('llm_qa' , llm_qa)

# add edges : 

graph.add_edge(START , 'llm_qa')
graph.add_edge('llm_qa' , END)

# compile graph :

workflow = graph.compile()


In [16]:
from google import genai

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explain how AI works in a few words"
)
print(interaction.output_text)

KeyboardInterrupt: 

In [ ]:
# Execute graph :

initial_state = {'question' : "what about demon slayer ?"}

final_output = workflow.invoke(initial_state)

print(final_output)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash-lite' (UNAUTHENTICATED): 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.', 'status': 'UNAUTHENTICATED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_TYPE_UNSUPPORTED', 'metadata': {'method': 'google.ai.generativelanguage.v1beta.GenerativeService.GenerateContent', 'service': 'generativelanguage.googleapis.com'}}]}}